# BFF - train the public datasets on a Colab GPU

Trains four models and exports them as ONNX for the backend:

| Model | Classes | Input | Data |
|---|---|---|---|
| `construction-vehicle-detector` | 8 vehicle types | 640 | Construction Vehicle Detection (Roboflow, CC BY 4.0) |
| `excavator-activity-detector` | 5 activity states | 640 | EXCAVATOR (Roboflow, CC BY 4.0) |
| `earthmoving-detector` | excavator, truck | 640 | end-to-end earthmoving archive |
| `earthmoving-activity-classifier` | idle, swing_bucket, load_bucket, dump, move (multi-label) | 224 | activity labels of the same archive |

Runtime on a T4: roughly 4-6 hours for the full set. Set `FAST = True`
for a 20 minute smoke run.

Before the first run: upload the three zips to
`My Drive/bff_datasets/` (see the last cell).

In [ ]:
# 1. GPU check.
!nvidia-smi
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
assert torch.cuda.is_available(), "pick a GPU runtime: Runtime > Change runtime type > GPU"

## 2. Clone the private repos

The repos are private. The cell below takes a GitHub token from the first
place it finds one:

1. `GITHUB_TOKEN` or `GITHUB_PAT` in the environment
2. the Colab secret of the same name
3. a `GITHUB_TOKEN=...` line in `/content/.env`
4. the `TOKEN` field in the cell

With a token the clone runs over HTTPS and no SSH key is needed. Without
one, a key that already exists in the runtime is used. Nothing is ever
deleted: a new key is never generated over an existing one.

In [ ]:
# 2. Clone the private repos.
#
# The repos are private. Two ways in:
#   a) a token -> HTTPS clone (works out of the box, no key needed)
#   b) an SSH key in the runtime -> SSH clone
# The token is looked up in this order:
#   GITHUB_TOKEN / GITHUB_PAT in the environment, the Colab secret,
#   a GITHUB_TOKEN= line in /content/.env, or the value pasted below.
#
# Colab secrets may be unavailable; paste the token into TOKEN below,
# or put it in a file and run:  %writefile /content/.env
TOKEN = ''          # <- paste the GitHub token here if nothing else works
import os, re, subprocess
from pathlib import Path


def find_token() -> str:
    for name in ('GITHUB_TOKEN', 'GITHUB_PAT'):
        value = os.environ.get(name, '').strip()
        if value:
            return value
    try:
        from google.colab import userdata
        for name in ('GITHUB_TOKEN', 'GITHUB_PAT'):
            value = (userdata.get(name) or '').strip()
            if value:
                return value
    except Exception as exc:
        print('colab secrets not usable:', exc)
    for candidate in (Path('/content/.env'), Path.home() / '.env'):
        if candidate.exists():
            match = re.search(r'^\s*(?:GITHUB_TOKEN|GITHUB_PAT)\s*=\s*(\S+)',
                              candidate.read_text(encoding='utf-8'), re.M)
            if match:
                return match.group(1).strip('"\'')
    return TOKEN.strip()


token = find_token()
print('token found:', bool(token))

ORG = 'mahardhika'
REF = 'main'
REPOS = {
    'bff-core': f'{ORG}/bff-core.git',
    'bff-ml':   f'{ORG}/bff-ml.git',
    'bff-ops':  f'{ORG}/bff-ops.git',
}

SSH_DIR = Path.home() / '.ssh'
SSH_DIR.mkdir(parents=True, exist_ok=True)
os.chmod(SSH_DIR, 0o700)
key = next((p for p in (SSH_DIR / 'id_ed25519', SSH_DIR / 'id_rsa')
            if p.exists()), None)
known_hosts = SSH_DIR / 'known_hosts'
with known_hosts.open('a') as handle:
    subprocess.run(['ssh-keyscan', '-H', 'github.com'], stdout=handle, check=False)
os.chmod(known_hosts, 0o600)
ssh_env = {**os.environ, 'GIT_SSH_COMMAND': 'ssh -o StrictHostKeyChecking=accept-new'}

for name, repo in REPOS.items():
    target = Path('/content') / name
    if target.exists():
        subprocess.run(['rm', '-rf', str(target)], check=True)
    if token:
        url = f'https://{token}@github.com/{repo}'
        print('clone over HTTPS with the token:', name)
        subprocess.run(['git', 'clone', '--branch', REF, '--depth', '1', url,
                        str(target)], check=True)
        # Keep the token out of the stored remote.
        subprocess.run(['git', '-C', str(target), 'remote', 'set-url', 'origin',
                        f'https://github.com/{repo}'], check=False)
    elif key is not None:
        print('clone over SSH:', name)
        subprocess.run(['git', 'clone', '--branch', REF, '--depth', '1',
                        f'git@github.com:{repo}', str(target)],
                       env=ssh_env, check=True)
    else:
        raise SystemExit(
            'No token and no SSH key. Paste a GitHub token into TOKEN above, '
            'or add a key to the runtime: !ssh-keygen -t ed25519 -N "" -f '
            '~/.ssh/id_ed25519 and add the public key to GitHub.')
print('cloned:', ', '.join(REPOS))

In [ ]:
# 3. Install the packages (bff-core first: bff-ml depends on it).
!pip install -q -e "/content/bff-core" -e "/content/bff-ml"
!python -c "import torch, ml; print('ml ok, torch', torch.__version__, torch.cuda.is_available())"

## 4. Data

The three zips come from Google Drive. Drive is slow for many small files,
so the archives are copied to the local disk first and unpacked there.

In [ ]:
# 4. Mount Drive and copy the archives to the local disk.
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ZIPS = Path('/content/drive/MyDrive/bff_datasets')
LOCAL_ZIPS = Path('/content/zips')
LOCAL_ZIPS.mkdir(parents=True, exist_ok=True)

zips = sorted(DRIVE_ZIPS.glob('*.zip'))
print('found on drive:', [z.name for z in zips])
assert zips, f'no zip in {DRIVE_ZIPS} - upload them first'

for z in zips:
    target = LOCAL_ZIPS / z.name
    if not target.exists():
        print('copying', z.name)
        subprocess.run(['cp', str(z), str(target)], check=True)
print('local copies:', sorted(p.name for p in LOCAL_ZIPS.glob('*.zip')))

In [ ]:
# 5. Unpack into the COCO training layout.
#    The archive names stay as they are on the laptop.
DATA_ROOT = Path('/content/datasets')
DATA_ROOT.mkdir(parents=True, exist_ok=True)
INGEST = '/content/bff-ops/scripts/ingest_dataset_zip.py'

PLAN = [
    ('EXCAVATOR.v1i.coco-mmdetection.zip', 'excavator-activity'),
    ('Construction Vehicle Detection.v2i.coco-mmdetection.zip', 'construction-vehicle'),
    ('Data for End-to-end visual detection, tracking and activity analysis of interacting earthmoving equipment.zip',
     'earthmoving-equipment'),
]
for archive, slug in PLAN:
    source = LOCAL_ZIPS / archive
    if not source.exists():
        print('missing archive, skipped:', archive)
        continue
    if (DATA_ROOT / slug / 'train' / 'annotations.json').exists():
        print('already ingested:', slug)
        continue
    subprocess.run(['python', INGEST, '--source', str(source),
                    '--slug', slug, '--dest', str(DATA_ROOT)], check=True)

for slug in [p[1] for p in PLAN]:
    root = DATA_ROOT / slug
    if root.exists():
        print(slug, sorted(p.name for p in root.iterdir()))

## 5. Train

`FAST = True` caps the images and the epochs. The full run trains on
everything.

In [ ]:
# 6. Train the four models on the GPU.
FAST = True        # False = full run
RUNS = Path('/content/runs')
RUNS.mkdir(parents=True, exist_ok=True)

EPOCHS_DET = 3 if FAST else 30
EPOCHS_CLS = 3 if FAST else 20
MAX_IMAGES = 2000 if FAST else 0        # 0 = whole dataset
BATCH = 16

JOBS = [
    dict(name='construction-vehicle-detector', kind='detector',
         data='/content/datasets/construction-vehicle/train',
         epochs=EPOCHS_DET, max_images=MAX_IMAGES, size=640),
    dict(name='excavator-activity-detector', kind='detector',
         data='/content/datasets/excavator-activity/train',
         epochs=EPOCHS_CLS, max_images=MAX_IMAGES, size=640),
    dict(name='earthmoving-detector', kind='detector',
         data='/content/datasets/earthmoving-equipment/train',
         epochs=EPOCHS_DET, max_images=MAX_IMAGES, size=640),
    dict(name='earthmoving-activity-classifier', kind='activity',
         data='/content/datasets/earthmoving-equipment',
         epochs=EPOCHS_CLS, max_images=MAX_IMAGES, size=224),
]

for job in JOBS:
    cmd = ['python', '-m', 'ml', 'train',
           '--data-source=local', f"--data-path={job['data']}",
           f"--model-name={job['name']}", '--version=0.1.0',
           f"--kind={job['kind']}", f"--epochs={job['epochs']}",
           f"--batch-size={BATCH}", '--device=cuda', '--workers=4',
           f"--output-dir={RUNS}", '--no-sync']
    if job['kind'] == 'detector':
        cmd.append(f"--input-size={job['size']}")
    else:
        cmd.append(f"--classifier-input-size={job['size']}")
    if job['max_images']:
        cmd.append(f"--max-images={job['max_images']}")
    print('\n===', job['name'], '===')
    subprocess.run(cmd, check=True)

In [ ]:
# 7. Collect the artifacts and store them on Drive.
import shutil, zipfile
from datetime import datetime

STAMP = datetime.now().strftime('%Y%m%d-%H%M')
OUT_ZIP = Path(f'/content/bff_runs_{STAMP}.zip')
DRIVE_OUT = Path('/content/drive/MyDrive/bff_runs')
DRIVE_OUT.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(OUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for run in sorted(RUNS.iterdir()):
        if not run.is_dir():
            continue
        for name in ('detector.onnx', 'classifier.onnx', 'labels.json',
                     'metrics.json', 'hparams.json', 'MODEL_CARD.md'):
            path = run / name
            if path.exists() and path.stat().st_size > 0:
                zf.write(path, f'{run.name}/{name}')
        print('collected', run.name)

shutil.copy(OUT_ZIP, DRIVE_OUT / OUT_ZIP.name)
print('saved to Drive:', DRIVE_OUT / OUT_ZIP.name)
print('size MB:', round(OUT_ZIP.stat().st_size / 1e6, 1))

## 6. Install the result on the laptop

1. Download `bff_runs_<stamp>.zip` from `My Drive/bff_runs/`.
2. Unzip it into `ml/runs/` of the workspace.
3. Copy the weights into the model cache, one folder per model:

```
data/models/construction-vehicle-detector/0.1.0/detector.onnx
data/models/construction-vehicle-detector/0.1.0/labels.json
data/models/excavator-activity-detector/0.1.0/detector.onnx
data/models/excavator-activity-detector/0.1.0/labels.json
data/models/earthmoving-detector/0.1.0/detector.onnx
data/models/earthmoving-detector/0.1.0/labels.json
data/models/earthmoving-activity-classifier/0.1.0/classifier.onnx
data/models/earthmoving-activity-classifier/0.1.0/labels.json
```

4. Choose the model in `.env`:

```
ACTIVE_DETECTOR_MODEL=construction-vehicle-detector@0.1.0
DETECTOR_KEEP_LABELS=Bulldozer,Dump Truck,Excavator,Grader,Loader,Mixer Truck,Mobile Crane,Roller
```

5. Restart the backend: `python -m backend`.